# Intelligent Web Crawler: Results Visualization & Evaluation

This notebook demonstrates how to evaluate UCS and A* using the shared project interface.
The included values are **dummy data for testing only**. Replace the sample graph and results
with outputs from Person 1 and Person 2 before reporting experimental findings.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.evaluation import (
    evaluate_results, evaluate_budget_experiments,
    save_metrics_csv, plot_metric_comparison, plot_budget_comparison
)

## 1. Create a small dummy graph

In [2]:
edges = [(i, i+1) for i in range(19)] + [(0, 5), (2, 10), (4, 15), (7, 18)]
rows, cols = zip(*edges)
adjacency = csr_matrix((np.ones(len(edges)), (rows, cols)), shape=(20, 20))

pagerank = np.array([
    0.15, 0.10, 0.08, 0.07, 0.06, 0.055, 0.05, 0.045, 0.04, 0.035,
    0.03, 0.028, 0.026, 0.024, 0.022, 0.02, 0.018, 0.016, 0.014, 0.012
], dtype=float)
pagerank = pagerank / pagerank.sum()

graph_data = {
    "adjacency": adjacency,
    "node_ids": np.arange(1000, 1020),
    "node_to_index": {node_id: i for i, node_id in enumerate(range(1000, 1020))},
    "in_degree": np.asarray(adjacency.sum(axis=0)).ravel(),
    "out_degree": np.asarray(adjacency.sum(axis=1)).ravel(),
    "pagerank": pagerank
}

## 2. Create dummy UCS and A* outputs

In [3]:
ucs_results = {
    "visited_node_indices": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    "visited_node_ids": [],
    "discovered_node_indices": [],
    "total_cost": 9,
    "execution_time": 0.017,
    "peak_memory_mb": 121.5
}
astar_results = {
    "visited_node_indices": [0, 5, 10, 15, 1, 2, 3, 4, 6, 7],
    "visited_node_ids": [],
    "discovered_node_indices": [],
    "total_cost": 9,
    "execution_time": 0.021,
    "peak_memory_mb": 123.2
}
comparison_df = evaluate_results(graph_data, ucs_results, astar_results, top_percent=1.0)
comparison_df

,Algorithm,Visited Webpages,Important Webpages Visited,PageRank Mass Captured,PageRank Mass Captured (%),Graph Coverage (%),Execution Time (seconds),Peak Memory (MB)
0,UCS,10,1,0.765363,76.536313,50.0,0.017,121.5
1,A*,10,1,0.737430,73.743017,50.0,0.021,123.2


## 3. Test multiple exploration budgets

In [4]:
budgets = [5, 10, 15, 20]
ucs_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
astar_order = [0, 5, 10, 15, 1, 2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 14, 16, 17, 18, 19]

experiment_results = {}
for budget in budgets:
    experiment_results[budget] = {
        "ucs": {
            "visited_node_indices": ucs_order[:budget],
            "execution_time": 0.0015 * budget + 0.002,
            "peak_memory_mb": 120 + 0.15 * budget
        },
        "astar": {
            "visited_node_indices": astar_order[:budget],
            "execution_time": 0.0018 * budget + 0.003,
            "peak_memory_mb": 122 + 0.12 * budget
        }
    }

budget_df = evaluate_budget_experiments(graph_data, experiment_results, top_percent=1.0)
budget_df

,Budget,Algorithm,Visited Webpages,Important Webpages Visited,PageRank Mass Captured,PageRank Mass Captured (%),Graph Coverage (%),Execution Time (seconds),Peak Memory (MB)
0,5,UCS,5,1,0.513966,51.396648,25.0,0.0095,120.75
1,5,A*,5,1,0.396648,39.664804,25.0,0.0120,122.60
2,10,UCS,10,1,0.765363,76.536313,50.0,0.0170,121.50
3,10,A*,10,1,0.737430,73.743017,50.0,0.0210,123.20
4,15,UCS,15,1,0.910615,91.061453,75.0,0.0245,122.25
5,15,A*,15,1,0.908380,90.837989,75.0,0.0300,123.80
6,20,UCS,20,1,1.000000,100.000000,100.0,0.0320,123.00
7,20,A*,20,1,1.000000,100.000000,100.0,0.0390,124.40


## 4. Export CSV and generate visualizations

In [5]:
output_dir = PROJECT_ROOT / "results"
figure_dir = output_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

save_metrics_csv(budget_df, output_dir / "metrics.csv")

metrics_to_plot = [
    ("Visited Webpages", "visited_webpages.png", "Unique webpages visited"),
    ("Important Webpages Visited", "important_pages.png", "Important webpages visited"),
    ("PageRank Mass Captured (%)", "pagerank_captured.png", "PageRank mass captured (%)"),
    ("Graph Coverage (%)", "graph_coverage.png", "Visited nodes / total nodes (%)"),
    ("Execution Time (seconds)", "execution_time.png", "Time (seconds)"),
    ("Peak Memory (MB)", "memory_consumption.png", "Process-wide peak RSS (MB)")
]

# A chart for each budget can be generated from one selected budget:
selected_budget = 10
selected = budget_df[budget_df["Budget"] == selected_budget]
for metric, filename, ylabel in metrics_to_plot:
    plot_metric_comparison(
        selected, metric, figure_dir / filename,
        title=f"UCS vs A*: {metric} (budget={selected_budget})",
        ylabel=ylabel
    )

# Budget trend charts are also useful for the report/poster.
for metric, filename, ylabel in metrics_to_plot:
    plot_budget_comparison(
        budget_df, metric, figure_dir / f"budget_{filename}",
        title=f"{metric} across exploration budgets",
        ylabel=ylabel
    )
print("CSV and charts saved.")

CSV and charts saved.


## 5. Measuring memory during real algorithm execution

The evaluation module measures process-wide resident set size (RSS). Wrap the call to
Person 2's algorithm like this, then store the measured value in its result dictionary:

```python
from src.evaluation import measure_peak_memory

with measure_peak_memory() as memory:
    ucs_results = run_ucs_or_crawler(...)

ucs_results["peak_memory_mb"] = memory.peak_memory_mb
```

Repeat for A*. The same process-wide measurement method must be used for both algorithms.
For a fair comparison, run algorithms sequentially, use the same machine/environment, and
record whether the Python process has a warm or cold cache.

## 6. Integration checklist

- `graph_data["pagerank"]` must align with internal node indices `0 ... N-1`.
- `visited_node_indices` must contain internal indices, not original SNAP IDs.
- Both algorithms must use the same graph, start node, budget and traversal costs.
- Use the same precomputed PageRank vector and the same top-PageRank threshold.
- Run each budget several times and compare mean (and preferably standard deviation).
- Do not interpret the dummy outputs as evidence about UCS or A*.